# Gold Mart Creation and Validation

This notebook creates and validates the final Gold dimensional model for the NYC Green Taxi project.

The model follows a star schema designed for trip-level analysis and business reporting.

%md
## Purpose

The purpose of this notebook is to transform the trusted Silver-layer data into reusable Gold dimensions and a trip-level fact table.

The final model will support analysis of trip dates, trip times, taxi zones, weather conditions, revenue, distance, duration, and passenger behavior.

%md
## Approved Gold Tables

The final Gold layer will contain the following tables:

- `dim_date`
- `dim_time`
- `dim_taxi_zone`
- `dim_weather_hour`
- `fact_green_taxi_trip`

The fact table grain is one row per Green Taxi trip.

%md
## Silver Source Inspection

The Gold layer will be created from the following Silver source tables:

- `green_taxi`
- `taxi_zones`
- `weather`

Before creating the Gold tables, we inspect the source structures and sample records to confirm the available columns and data types.

In [0]:
SHOW TABLES IN `ftw-week-08`.`02_silver`;

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.green_taxi;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.green_taxi
LIMIT 10;

%md
## Green Taxi Table Structure

The `green_taxi` Silver table contains the trip-level source data.

It provides the pickup and dropoff timestamps, pickup and dropoff location identifiers, trip distance, trip duration, passenger count, and financial measures such as fare, tip, tolls, and total amount.

This table will become the main source for `fact_green_taxi_trip`.

%md
## Green Taxi Column Inventory

The Green Taxi columns were reviewed to identify the fields needed for the fact table.

The important fields include the trip timestamps, location identifiers, trip measures, financial measures, and `batch_id` for lineage.

The source does not provide a guaranteed unique trip identifier, so a deterministic `trip_key` will be generated from stable trip attributes.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.taxi_zones;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.taxi_zones
LIMIT 10;

%md
## Taxi Zone Table Structure

The `taxi_zones` table provides the descriptive information for each taxi location identifier.

This table will be transformed into `dim_taxi_zone`, which will be used for both pickup-zone and dropoff-zone analysis.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.weather;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.weather
ORDER BY weather_datetime
LIMIT 10;

%md
## Weather Column Inventory and Preview

The `weather` table contains hourly weather observations identified by `weather_datetime`.

The available weather attributes include temperature, precipitation, rain, snowfall, weather code, and wind speed.

The weather dimension will be joined to trips using the pickup time rounded to the matching weather hour.

%md
## Final Gold Dimensional Model

The approved star schema contains four dimensions and one fact table.

The fact table stores one row per Green Taxi trip. The dimensions provide reusable descriptive attributes for dates, times, taxi zones, and hourly weather.

Pickup and dropoff dates, times, and zones will be treated as role-playing dimensions in the fact table.

%md%md
## Existing Gold Table Assessment

The existing Gold schema contains tables from the previous design:

- `dim_datetime`
- `dim_location`
- `dim_weather`
- `fact_taxi_trips`

These tables are retained for reference and will not be used in the final implementation because their design does not match the newly approved star schema.

The final implementation will use the new tables:

- `dim_date`
- `dim_time`
- `dim_taxi_zone`
- `dim_weather_hour`
- `fact_green_taxi_trip`

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_date
USING DELTA
AS
WITH date_range AS (
    SELECT
        MIN(TO_DATE(lpep_pickup_datetime)) AS min_date,
        MAX(TO_DATE(lpep_pickup_datetime)) AS max_date
    FROM `ftw-week-08`.`02_silver`.green_taxi
),
dates AS (
    SELECT EXPLODE(
        SEQUENCE(min_date, max_date, INTERVAL 1 DAY)
    ) AS full_date
    FROM date_range
)
SELECT
    CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT) AS date_key,
    full_date,
    YEAR(full_date) AS year,
    QUARTER(full_date) AS quarter,
    MONTH(full_date) AS month_number,
    DATE_FORMAT(full_date, 'MMMM') AS month_name,
    DAYOFMONTH(full_date) AS day_of_month,
    DATE_FORMAT(full_date, 'EEEE') AS day_name,
    DAYOFWEEK(full_date) AS day_of_week,
    CASE
        WHEN DAYOFWEEK(full_date) IN (1, 7) THEN TRUE
        ELSE FALSE
    END AS is_weekend,
    FALSE AS is_holiday
FROM dates;

%md
## Create DIM_DATE

The `dim_date` table was created from the minimum and maximum Green Taxi pickup dates.

A complete calendar row was generated for every date in that range. Each date has a unique `date_key` and descriptive calendar attributes such as year, quarter, month, day, weekend indicator, and holiday indicator.

In [0]:
SELECT
    COUNT(*) AS row_count,
    MIN(full_date) AS minimum_date,
    MAX(full_date) AS maximum_date,
    COUNT(DISTINCT date_key) AS unique_date_keys
FROM `ftw-week-08`.`03_gold`.dim_date;

In [0]:
SELECT *
FROM `ftw-week-08`.`03_gold`.dim_date
ORDER BY full_date
LIMIT 10;

In [0]:
DESCRIBE TABLE `ftw-week-08`.`03_gold`.dim_date;

%md
## DIM_DATE Validation Result

The `dim_date` table was successfully created and validated.

The table contains 6,362 rows and 6,362 unique date keys. The date range covers 2008-12-31 to 2026-06-01.

The preview and schema inspection confirmed that the expected date attributes and data types are present. The dimension is ready to support the Gold fact table.

In [0]:
%md%md
## Create DIM_TIME

The `dim_time` table contains one row for every hour of the day.

It provides reusable time attributes for pickup-hour and dropoff-hour analysis. Its expected grain is one row per hour.


In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_time
USING DELTA
AS
SELECT
    hour_number AS time_key,
    hour_number,
    CONCAT(
        LPAD(CAST(hour_number AS STRING), 2, '0'),
        ':00'
    ) AS time_label,
    CASE
        WHEN hour_number = 0 THEN 12
        WHEN hour_number <= 12 THEN hour_number
        ELSE hour_number - 12
    END AS hour_12,
    CASE
        WHEN hour_number < 12 THEN 'AM'
        ELSE 'PM'
    END AS am_pm,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM (
    SELECT EXPLODE(SEQUENCE(0, 23)) AS hour_number
);

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT time_key) AS unique_time_keys,
    MIN(time_key) AS minimum_time_key,
    MAX(time_key) AS maximum_time_key
FROM `ftw-week-08`.`03_gold`.dim_time;

%md
## Gold Mart Preparation
The initial Gold mart preparation was completed using the approved star schema design.

The Silver-layer source tables were inspected and documented, including:

- `green_taxi`
- `taxi_zones`
- `weather`

The existing Gold tables were retained for reference and were not modified.

The new `dim_date` table was successfully created and validated with 6,362 rows and 6,362 unique date keys. The date range covers 2008-12-31 to 2026-06-01.

The `dim_time` table was also created with 24 unique hourly records, covering hours 00:00 to 23:00.

The next implementation steps are to create and validate:

- `dim_taxi_zone`
- `dim_weather_hour`
- `fact_green_taxi_trip`

The final fact table will follow the approved grain:

One row represents one Green Taxi trip.

The completed notebook and Gold-layer changes were saved and pushed to the feature branch for team review.

## Create DIM_TAXI_ZONE

The `dim_taxi_zone` table provides descriptive information about each taxi zone.

It will be used as a reusable dimension for both pickup and dropoff locations in the Green Taxi fact table.

Expected grain:

One row represents one taxi zone.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.taxi_zones;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.taxi_zones
LIMIT 10;

%md
## Taxi Zone Source Review

The taxi zone source table was inspected to confirm its available columns and structure.

The location identifier will be used as the primary key of `dim_taxi_zone`. The descriptive zone attributes will support pickup and dropoff location analysis.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_taxi_zone
USING DELTA
AS
SELECT DISTINCT
    CAST(LocationID AS INT) AS taxi_zone_key,
    CAST(LocationID AS INT) AS location_id,
    Borough AS borough,
    Zone AS zone_name,
    service_zone,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM `ftw-week-08`.`02_silver`.taxi_zones
WHERE LocationID IS NOT NULL;

%md
## Validate DIM_TAXI_ZONE

The `dim_taxi_zone` table will be validated by checking the total number of records, unique taxi zone keys, and null keys.

Each taxi zone should have one unique key, and the key should not be null.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT taxi_zone_key) AS unique_taxi_zone_keys,
    SUM(
        CASE
            WHEN taxi_zone_key IS NULL THEN 1
            ELSE 0
        END
    ) AS null_key_count
FROM `ftw-week-08`.`03_gold`.dim_taxi_zone;

%md
## DIM_TAXI_ZONE Validation Result

The `dim_taxi_zone` table was created successfully from the Silver taxi zone source.

The validation confirms that the taxi zone keys are unique and that no null taxi zone keys are present.

The dimension is ready to support pickup-zone and dropoff-zone relationships in the Gold fact table.

%md
## Create DIM_WEATHER_HOUR

The `dim_weather_hour` table stores one row for each hourly weather observation.

It will provide weather attributes such as temperature, precipitation, rain, snowfall, weather code, and wind speed.

The weather timestamp will be used as the unique hourly key for joining weather conditions to Green Taxi trips.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.weather;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.weather
ORDER BY weather_datetime
LIMIT 10;

%md
## Weather Source Review

The Silver weather table was inspected to confirm the timestamp and weather measurement columns.

The `weather_datetime` column represents the hourly observation time and will be used to create the weather dimension key.

The weather measurements will be retained as descriptive attributes for trip and weather analysis.

%md
## Create DIM_WEATHER_HOUR

The `dim_weather_hour` table stores one record for each hourly weather observation.

The `weather_datetime` column is used as the unique weather-hour key. Weather measurements such as temperature, precipitation, rain, snowfall, weather code, and wind speed are retained for analysis.

Expected grain:

One row represents one hourly weather observation.

In [0]:
SHOW TABLES IN `ftw-week-08`.`03_gold`;

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_weather_hour
USING DELTA
AS
SELECT
    weather_datetime,
    TO_DATE(weather_datetime) AS weather_date,
    HOUR(weather_datetime) AS weather_hour,
    temperature_2m,
    precipitation,
    rain,
    snowfall,
    weather_code,
    wind_speed_10m,
    CURRENT_TIMESTAMP() AS gold_created_at
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY weather_datetime
            ORDER BY weather_datetime
        ) AS row_num
    FROM `ftw-week-08`.`02_silver`.weather
)
WHERE row_num = 1;

%md
## Validate DIM_WEATHER_HOUR

The weather dimension will be validated by checking the total number of hourly records, unique weather timestamps, and null weather keys.

Each `weather_datetime` should represent one unique hourly observation.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT weather_datetime) AS unique_weather_hours,
    SUM(
        CASE
            WHEN weather_datetime IS NULL THEN 1
            ELSE 0
        END
    ) AS null_key_count
FROM `ftw-week-08`.`03_gold`.dim_weather_hour;

In [0]:
SELECT *
FROM `ftw-week-08`.`03_gold`.dim_weather_hour
ORDER BY weather_datetime
LIMIT 10;

%md
## DIM_WEATHER_HOUR Validation Result

The `dim_weather_hour` table was created from the Silver weather source.

Duplicate weather timestamps were removed so that each weather hour has one record. The validation checks that weather timestamps are unique and that no null weather keys are present.

The dimension is ready to support the weather relationship in the Gold fact table.

%md
## Create FACT_GREEN_TAXI_TRIP

The `fact_green_taxi_trip` table stores the measurable business events from the Green Taxi trips.

Expected grain:

One row represents one Green Taxi trip.

The fact table will contain trip timestamps, pickup and dropoff keys, weather reference, trip measures, financial measures, and `batch_id` for lineage.

A deterministic `trip_key` will be generated because the source does not provide a guaranteed unique trip identifier.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.green_taxi;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.green_taxi
LIMIT 10;

%md
## Create FACT_GREEN_TAXI_TRIP

The `fact_green_taxi_trip` table stores one record for each Green Taxi trip.

The deterministic `trip_key` is generated using stable trip attributes: VendorID, pickup and dropoff timestamps, pickup location, and dropoff location.

The fact table connects to the date, time, taxi zone, and weather dimensions through foreign keys.

Expected grain:

One row represents one Green Taxi trip.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.fact_green_taxi_trip
USING DELTA
AS
SELECT
    xxhash64(
        CAST(g.VendorID AS STRING),
        CAST(g.lpep_pickup_datetime AS STRING),
        CAST(g.lpep_dropoff_datetime AS STRING),
        CAST(g.PULocationID AS STRING),
        CAST(g.DOLocationID AS STRING)
    ) AS trip_key,

    g.VendorID AS vendor_id,

    g.lpep_pickup_datetime AS pickup_datetime,
    g.lpep_dropoff_datetime AS dropoff_datetime,

    CAST(DATE_FORMAT(TO_DATE(g.lpep_pickup_datetime), 'yyyyMMdd') AS INT)
        AS pickup_date_key,

    CAST(DATE_FORMAT(TO_DATE(g.lpep_dropoff_datetime), 'yyyyMMdd') AS INT)
        AS dropoff_date_key,

    HOUR(g.lpep_pickup_datetime) AS pickup_time_key,
    HOUR(g.lpep_dropoff_datetime) AS dropoff_time_key,

    CAST(g.PULocationID AS INT) AS pickup_taxi_zone_key,
    CAST(g.DOLocationID AS INT) AS dropoff_taxi_zone_key,

    DATE_TRUNC('HOUR', g.lpep_pickup_datetime) AS weather_datetime,

    g.passenger_count,
    g.trip_distance,
    g.trip_duration_minutes,

    g.fare_amount,
    g.tip_amount,
    g.tolls_amount,
    g.total_amount,

    g.payment_type,
    g.trip_type,
    g.congestion_surcharge,

    g.batch_id,

    CURRENT_TIMESTAMP() AS gold_created_at

FROM `ftw-week-08`.`02_silver`.green_taxi AS g;

%md
## Validate FACT_GREEN_TAXI_TRIP Grain

The fact table will be validated according to its approved grain: one row represents one Green Taxi trip.

We will check the total number of records, unique trip keys, duplicate trip keys, and null trip keys.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT trip_key) AS unique_trip_keys,
    COUNT(*) - COUNT(DISTINCT trip_key) AS duplicate_trip_count,
    SUM(
        CASE
            WHEN trip_key IS NULL THEN 1
            ELSE 0
        END
    ) AS null_trip_key_count
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip;

%md
## FACT_GREEN_TAXI_TRIP Duplicate Investigation

The fact table contains 133,367 rows and 132,983 unique trip keys.

There are 384 duplicate trip keys. This indicates that some source records share the same trip-defining attributes or that duplicate source records were included.

The duplicates must be investigated and removed so that the fact table follows its approved grain of one row per Green Taxi trip.

In [0]:
SELECT
    trip_key,
    COUNT(*) AS duplicate_count
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip
GROUP BY trip_key
HAVING COUNT(*) > 1
ORDER BY duplicate_count DESC
LIMIT 20;

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.fact_green_taxi_trip
USING DELTA
AS
WITH source_deduplicated AS (
    SELECT
        g.*,
        ROW_NUMBER() OVER (
            PARTITION BY
                VendorID,
                lpep_pickup_datetime,
                lpep_dropoff_datetime,
                PULocationID,
                DOLocationID
            ORDER BY batch_id
        ) AS row_num
    FROM `ftw-week-08`.`02_silver`.green_taxi AS g
)
SELECT
    xxhash64(
        CAST(VendorID AS STRING),
        CAST(lpep_pickup_datetime AS STRING),
        CAST(lpep_dropoff_datetime AS STRING),
        CAST(PULocationID AS STRING),
        CAST(DOLocationID AS STRING)
    ) AS trip_key,

    VendorID AS vendor_id,

    lpep_pickup_datetime AS pickup_datetime,
    lpep_dropoff_datetime AS dropoff_datetime,

    CAST(DATE_FORMAT(TO_DATE(lpep_pickup_datetime), 'yyyyMMdd') AS INT)
        AS pickup_date_key,

    CAST(DATE_FORMAT(TO_DATE(lpep_dropoff_datetime), 'yyyyMMdd') AS INT)
        AS dropoff_date_key,

    HOUR(lpep_pickup_datetime) AS pickup_time_key,
    HOUR(lpep_dropoff_datetime) AS dropoff_time_key,

    CAST(PULocationID AS INT) AS pickup_taxi_zone_key,
    CAST(DOLocationID AS INT) AS dropoff_taxi_zone_key,

    DATE_TRUNC('HOUR', lpep_pickup_datetime) AS weather_datetime,

    passenger_count,
    trip_distance,
    trip_duration_minutes,

    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,

    payment_type,
    trip_type,
    congestion_surcharge,
    batch_id,

    CURRENT_TIMESTAMP() AS gold_created_at

FROM source_deduplicated
WHERE row_num = 1;

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT trip_key) AS unique_trip_keys,
    COUNT(*) - COUNT(DISTINCT trip_key) AS duplicate_trip_count,
    SUM(
        CASE
            WHEN trip_key IS NULL THEN 1
            ELSE 0
        END
    ) AS null_trip_key_count
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip;

%md
## Validate FACT Foreign-Key Relationships

The fact table will be checked against the Gold dimensions to confirm that every foreign key has a matching dimension record.

This validates the relationships for:

- Pickup and dropoff dates
- Pickup and dropoff times
- Pickup and dropoff taxi zones
- Pickup weather hour

Unmatched keys may indicate missing dimension records or incorrect source values.

In [0]:
SELECT
    SUM(CASE WHEN d_pickup.date_key IS NULL THEN 1 ELSE 0 END)
        AS missing_pickup_date_keys,

    SUM(CASE WHEN d_dropoff.date_key IS NULL THEN 1 ELSE 0 END)
        AS missing_dropoff_date_keys,

    SUM(CASE WHEN t_pickup.time_key IS NULL THEN 1 ELSE 0 END)
        AS missing_pickup_time_keys,

    SUM(CASE WHEN t_dropoff.time_key IS NULL THEN 1 ELSE 0 END)
        AS missing_dropoff_time_keys,

    SUM(CASE WHEN z_pickup.taxi_zone_key IS NULL THEN 1 ELSE 0 END)
        AS missing_pickup_zone_keys,

    SUM(CASE WHEN z_dropoff.taxi_zone_key IS NULL THEN 1 ELSE 0 END)
        AS missing_dropoff_zone_keys,

    SUM(CASE WHEN w.weather_datetime IS NULL THEN 1 ELSE 0 END)
        AS missing_weather_keys

FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f

LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS d_pickup
    ON f.pickup_date_key = d_pickup.date_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS d_dropoff
    ON f.dropoff_date_key = d_dropoff.date_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t_pickup
    ON f.pickup_time_key = t_pickup.time_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t_dropoff
    ON f.dropoff_time_key = t_dropoff.time_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z_pickup
    ON f.pickup_taxi_zone_key = z_pickup.taxi_zone_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z_dropoff
    ON f.dropoff_taxi_zone_key = z_dropoff.taxi_zone_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
    ON f.weather_datetime = w.weather_datetime;

%md
## Foreign-Key Validation Result

The fact table was checked against all Gold dimensions.

A result of zero for all missing-key checks confirms that the fact table can successfully connect to the date, time, taxi zone, and weather dimensions.

%md
## Reconcile Gold Fact with Silver Source

The Gold fact table will be reconciled with the Silver Green Taxi source.

We will compare the number of unique trips and the total revenue between both layers.

This confirms that the Gold transformation preserved the expected business records and financial totals after removing duplicate trip records.

In [0]:
WITH silver_trip_summary AS (
    SELECT
        COUNT(DISTINCT
            xxhash64(
                CAST(VendorID AS STRING),
                CAST(lpep_pickup_datetime AS STRING),
                CAST(lpep_dropoff_datetime AS STRING),
                CAST(PULocationID AS STRING),
                CAST(DOLocationID AS STRING)
            )
        ) AS silver_unique_trips,
        SUM(total_amount) AS silver_total_revenue
    FROM `ftw-week-08`.`02_silver`.green_taxi
),

gold_trip_summary AS (
    SELECT
        COUNT(DISTINCT trip_key) AS gold_unique_trips,
        SUM(total_amount) AS gold_total_revenue
    FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip
)

SELECT
    silver_unique_trips,
    gold_unique_trips,
    silver_unique_trips - gold_unique_trips AS trip_count_difference,

    silver_total_revenue,
    gold_total_revenue,
    silver_total_revenue - gold_total_revenue AS revenue_difference
FROM silver_trip_summary
CROSS JOIN gold_trip_summary;

%md
## Reconciliation Result

The Gold fact table was compared with the Silver Green Taxi source using the approved deterministic trip key.

The reconciliation documents the trip-count and revenue differences caused by removing duplicate trip records.

The Gold layer prioritizes a reliable one-row-per-trip grain, unique trip keys, valid dimension relationships, and traceable lineage through `batch_id`.

%md
## Business Question 1: Revenue by Taxi Zone

This query identifies the pickup taxi zones that generate the highest total revenue.

It uses the fact table for financial measures and `dim_taxi_zone` for descriptive zone names.

In [0]:
SELECT
    z.zone_name AS pickup_zone,
    z.borough,
    COUNT(*) AS total_trips,
    ROUND(SUM(f.total_amount), 2) AS total_revenue
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z
    ON f.pickup_taxi_zone_key = z.taxi_zone_key
GROUP BY
    z.zone_name,
    z.borough
ORDER BY total_revenue DESC
LIMIT 10;

%md
## Business Question 1 Result

The results identify the pickup taxi zones with the highest number of trips and total revenue.

This shows which locations contribute most to Green Taxi activity and revenue. The combination of `fact_green_taxi_trip` and `dim_taxi_zone` allows the results to be grouped using readable zone and borough names.

%md
## Business Question 2: Trip Demand by Hour

This query shows the number of Green Taxi trips by pickup hour.

It uses `dim_time` to provide readable time labels for hourly demand analysis.

In [0]:
SELECT
    t.time_key AS pickup_hour,
    t.time_label,
    t.am_pm,
    COUNT(*) AS total_trips,
    ROUND(SUM(f.total_amount), 2) AS total_revenue
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t
    ON f.pickup_time_key = t.time_key
GROUP BY
    t.time_key,
    t.time_label,
    t.am_pm
ORDER BY pickup_hour;

%md
## Business Question 2 Result

The results show the number of Green Taxi trips and total revenue for each pickup hour.

This helps identify the hours with the highest trip demand and compare activity across the day. The `dim_time` table provides readable time labels and AM/PM classifications for reporting.

%md
## Business Question 3: Trips by Weather Condition

This query examines trip activity alongside hourly weather conditions.

It combines the Green Taxi fact table with `dim_weather_hour` using the pickup weather hour.

In [0]:
SELECT
    w.weather_code,
    COUNT(*) AS total_trips,
    ROUND(AVG(w.temperature_2m), 2) AS average_temperature,
    ROUND(SUM(f.total_amount), 2) AS total_revenue
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
    ON f.weather_datetime = w.weather_datetime
GROUP BY w.weather_code
ORDER BY total_trips DESC;

%md
## Business Question 3 Result

The results show the number of Green Taxi trips and total revenue associated with each weather condition.

This analysis demonstrates how the Gold star schema can combine trip-level facts with hourly weather attributes for business analysis.

%md
## Final Gold Mart Conclusion

The Gold dimensional model was successfully created from the approved Silver sources.

The completed Gold tables are:

- `dim_date`
- `dim_time`
- `dim_taxi_zone`
- `dim_weather_hour`
- `fact_green_taxi_trip`

The fact table follows the approved grain of one row per Green Taxi trip.

All primary validations passed:

- Date keys are unique
- Time dimension contains 24 hourly records
- Taxi zone keys are unique and non-null
- Weather timestamps are unique and non-null
- Trip keys are unique and non-null
- All foreign-key validation results are zero
- Gold and Silver unique trip counts match at 132,983

The Gold layer is ready to support downstream analytics, dashboards, and business-question analysis.